# Entrenamiento End-to-End Múltiescala + KAN Head (Pipeline de 3 Fases)

Este notebook ejecuta el pipeline óptimo para transfer learning con una nueva cabeza:

1. **Fase 1 (Pre-entrenamiento del Backbone)**: Entrenar el extractor de características con una cabeza simple para aprender las representaciones generales.
2. **Fase 2 (Congelar y Educar KAN)**: Reemplazar la cabeza por KAN, congelar el backbone y entrenar solo el KAN para que se acople a las features aprendidas.
3. **Fase 3 (Fine-Tuning Conjunto)**: Descongelar los últimos bloques y entrenar todo a lr muy bajo para el ajuste final.
4. **Threshold Moving**: Calibración de umbrales óptimos por clase.

In [ ]:
!pip install -q medmnist monai

In [ ]:
!git clone https://github.com/ZiyaoLi/fast-kan

# 2. Entrar al directorio y realizar la instalación
%cd fast-kan
!pip install .

fatal: destination path 'fast-kan' already exists and is not an empty directory.
/content/fast-kan
Processing /content/fast-kan
  Preparing metadata (setup.py) ... done
  Created wheel for fastkan: filename=fastkan-0.0.1-py3-none-any.whl size=11558 sha256=110e1b5d955bdd0d5d32c62e5655e327da3f225b14d15ba9bd2665c1f1c5298f
  Stored in directory: /root/.cache/pip/wheels/f0/04/a4/adaeb274c8f8955e005051f68a2d3711dd0cc9e32682e96c18
Successfully built fastkan
  Attempting uninstall: fastkan
    Found existing installation: fastkan 0.0.1
    Uninstalling fastkan-0.0.1:
      Successfully uninstalled fastkan-0.0.1


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import monai.transforms as mt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from google.colab import drive
import pickle
from tqdm import tqdm

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
# Instalar fastkan (GPU-native, rapido)
import subprocess
from fastkan import FastKAN as KAN

## 1. Configuración del Entorno y Carga de Configuraciones previas

Primero necesitamos montar Drive y cargar las estadísticas y pesos que calculamos en la Fase 1.

In [ ]:
# Montar Drive si estamos en Colab
try:
    drive.mount('/content/drive')
    print("Drive montado con éxito.")
except:
    print("Ejecutando en entorno local.")

# Rutas Base
BASE_DIR = "/content/drive/MyDrive/Clasificacion_Lesiones_Dermatologicas"
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Cargar configuraciones (Pre-calculadas en Fase 1 y 2)
try:
    train_stats = np.load(os.path.join(DATA_DIR, "train_pixel_stats.npy"), allow_pickle=True).item()
    class_weights_np = np.load(os.path.join(DATA_DIR, "class_weights.npy"))
    print("Estadísticas y pesos cargados correctamente.")
except FileNotFoundError:
    print("ERROR: Faltan archivos de la Fase 1 (train_pixel_stats.npy o class_weights.npy).")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado con éxito.
Estadísticas y pesos cargados correctamente.
Usando dispositivo: cuda


## 2. Re-instanciación Standalone

Para que este notebook pueda ejecutarse solo en Colab, necesitamos redefinir rápidamente el Wrapper del Dataset, las transformaciones, y nuestra arquitectura `LesionClassifier`.

In [ ]:

# --- 2.1 Dataset y Transforms (augmentación más fuerte solo en minoritarias) ---
import medmnist
from medmnist import DermaMNIST

class DermaMNISTWithTransforms(Dataset):
    def __init__(self, split, root, transform=None, transform_soft=None, transform_strong_by_class=None):
        self.dataset = DermaMNIST(split=split, download=True, root=root, size=28)
        self.transform = transform
        self.transform_soft = transform_soft
        self.transform_strong_by_class = transform_strong_by_class or {}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img_np = np.array(img)
        label_val = int(label.item()) if hasattr(label, 'item') else int(label)
        if self.transform is not None:
            img_t = self.transform(img_np)
        else:
            if label_val in self.transform_strong_by_class:
                img_t = self.transform_strong_by_class[label_val](img_np)
            else:
                img_t = self.transform_soft(img_np)
        label_t = torch.tensor(label_val, dtype=torch.long)
        return img_t, label_t

# Normalización solo por imagen: Max-RGB (percentil 99) + min-max a [0,1]. Sin media/std global.
def _max_rgb_then_minmax(img):
    """Por imagen: White-patch (p99 por canal) + reescalado min-max a [0,1]. Evita dominantes por media global."""
    eps = 1e-8
    out = np.asarray(img, dtype=np.float64)
    if out.ndim != 3:
        return out.astype(np.float32)
    # Max-RGB: por canal, dividir por percentil 99 para blanquear la zona más clara
    for c in range(out.shape[0]):
        chan = out[c]
        p99 = np.percentile(chan, 99)
        if p99 > eps:
            out[c] = chan / p99
    out = np.clip(out, 0, None)
    # Min-max por imagen a [0,1]
    mn, mx = out.min(), out.max()
    if mx - mn > eps:
        out = (out - mn) / (mx - mn)
    return np.clip(out, 0, 1).astype(np.float32)

_max_rgb_minmax = mt.Lambda(func=_max_rgb_then_minmax)

# Pipelines MONAI: soft (nv) + fuertes por minoritaria (akiec, df, bcc/bkl/mel/vasc)
def _train_base():
    return [
        mt.EnsureChannelFirst(channel_dim=-1),
        mt.ScaleIntensityRange(a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
        _max_rgb_minmax,
    ]
def _train_end():
    return [
        mt.Resize(spatial_size=(28, 28)),
    ]
train_transforms_soft = mt.Compose(_train_base() + [
    mt.RandRotate(range_x=np.pi/4, prob=0.5, keep_size=True, padding_mode='reflection'),
    mt.RandFlip(spatial_axis=0, prob=0.5),
    mt.RandFlip(spatial_axis=1, prob=0.5),
    mt.RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.5, keep_size=True),
    mt.RandGaussianNoise(prob=0.2, std=0.05),
    mt.RandAdjustContrast(prob=0.3, gamma=(0.8, 1.2)),
] + _train_end())
train_transforms_akiec = mt.Compose(_train_base() + [
    mt.RandRotate(range_x=25 * np.pi / 180, prob=0.6, keep_size=True, padding_mode='reflection'),
    mt.RandFlip(spatial_axis=0, prob=0.6),
    mt.RandFlip(spatial_axis=1, prob=0.6),
    mt.RandZoom(min_zoom=0.92, max_zoom=1.08, prob=0.5, keep_size=True),
    mt.RandGaussianNoise(prob=0.25, std=0.04),
    mt.RandAdjustContrast(prob=0.35, gamma=(0.88, 1.12)),
] + _train_end())
train_transforms_df = mt.Compose(_train_base() + [
    mt.RandRotate(range_x=35 * np.pi / 180, prob=0.7, keep_size=True, padding_mode='reflection'),
    mt.RandFlip(spatial_axis=0, prob=0.7),
    mt.RandFlip(spatial_axis=1, prob=0.7),
    mt.RandZoom(min_zoom=0.88, max_zoom=1.12, prob=0.6, keep_size=True),
    mt.RandGaussianNoise(prob=0.35, std=0.05),
    mt.RandAdjustContrast(prob=0.4, gamma=(0.82, 1.18)),
] + _train_end())
train_transforms_intermediate = mt.Compose(_train_base() + [
    mt.RandRotate(range_x=30 * np.pi / 180, prob=0.65, keep_size=True, padding_mode='reflection'),
    mt.RandFlip(spatial_axis=0, prob=0.65),
    mt.RandFlip(spatial_axis=1, prob=0.65),
    mt.RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.55, keep_size=True),
    mt.RandGaussianNoise(prob=0.3, std=0.05),
    mt.RandAdjustContrast(prob=0.38, gamma=(0.85, 1.15)),
] + _train_end())
transform_strong_by_class = {0: train_transforms_akiec, 1: train_transforms_intermediate, 2: train_transforms_intermediate, 3: train_transforms_df, 4: train_transforms_intermediate, 6: train_transforms_intermediate}

val_test_transforms = mt.Compose([
    mt.EnsureChannelFirst(channel_dim=-1),
    mt.ScaleIntensityRange(a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    _max_rgb_minmax,
    mt.Resize(spatial_size=(28, 28)),
])

# DataLoaders — train con augmentación fuerte por minoritarias; val/test sin augmentación
BATCH_SIZE = 64
train_dataset = DermaMNISTWithTransforms('train', DATA_DIR, transform_soft=train_transforms_soft, transform_strong_by_class=transform_strong_by_class)
val_dataset = DermaMNISTWithTransforms('val', DATA_DIR, transform=val_test_transforms)
test_dataset = DermaMNISTWithTransforms('test', DATA_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print('Train: augmentación fuerte por minoritarias (akiec, df, bcc, bkl, mel, vasc); val/test sin augmentation.')

Train: 7007 | Val: 1003 | Test: 2005
Train: augmentación fuerte por minoritarias (akiec, df, bcc, bkl, mel, vasc); val/test sin augmentation.


In [ ]:
from fastkan import FastKAN as KAN

# --- 2.2 Reinstanciación del Modelo Múltiescala ---
class EuclideanFusion(nn.Module):
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon
    def forward(self, x1, x2):
        return torch.sqrt(torch.pow(x1, 2) + torch.pow(x2, 2) + self.epsilon)

class HadamardMaxFusion(nn.Module):
    def forward(self, x1, x2):
        return (x1 * x2) + torch.max(x1, x2)

class PseudoRadialFusion(nn.Module):
    def forward(self, x1, x2):
        diff_sq = torch.pow(x1 - x2, 2)
        similarity = torch.exp(-diff_sq / 2.0)
        return similarity * (x1 + x2)

class CrossAttentionFusion(nn.Module):
    def forward(self, x1, x2):
        return x1 * torch.sigmoid(x2)

class MultiscaleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, fusion_module, stride=1):
        super().__init__()
        # Ramas originales intactas
        self.conv3x3 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(True))
        self.conv5x5 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 5, stride, 2, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(True))
        self.fusion = fusion_module
        # --- EL ADN MedViTV2: Conexión Residual (Shortcut) ---
        self.shortcut = nn.Sequential()
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.final_relu = nn.ReLU(True)

    def forward(self, x):
        # 1. Extracción paralela
        x1 = self.conv3x3(x)
        x2 = self.conv5x5(x)
        # 2. Fusión matemática compleja
        fused = self.fusion(x1, x2)
        out = fused + self.shortcut(x)
        return self.final_relu(out)

# ==========================================
# NUEVA ARQUITECTURA: Backbone + KAN Nativos
# ==========================================
class RobustHead(nn.Module):
    def __init__(self, in_features=25, num_classes=7):
        super().__init__()
        # 1. Capa densa: expande las 25 features del backbone
        self.fc_pre = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        # 2. FastKAN: red KAN con funciones aprendibles, GPU-optimizada
        self.kan = KAN([128, 64])
        # 3. Clasificador final
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.fc_pre(x)
        x = self.kan(x)
        return self.head(x)

class LesionClassifierKAN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.block1 = MultiscaleBlock(3, 50, EuclideanFusion())
        self.pool1 = nn.MaxPool2d(2)
        self.block2 = MultiscaleBlock(50, 100, HadamardMaxFusion())
        self.pool2 = nn.MaxPool2d(2)
        self.block3 = MultiscaleBlock(100, 75, PseudoRadialFusion())
        self.pool3 = nn.MaxPool2d(2)
        self.block4 = MultiscaleBlock(75, 25, CrossAttentionFusion())
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # La cabeza puede ser intercambiable. Por defecto iniciamos con KAN
        self.classifier = RobustHead(in_features=25, num_classes=num_classes)

    def forward(self, x):
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.pool3(self.block3(x))
        x = self.gap(self.block4(x))
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = LesionClassifierKAN(num_classes=7).to(device)
print("Modelo LesionClassifierKAN instanciado correctamente.")

Modelo LesionClassifierKAN instanciado correctamente.


### 2.3 Congelamiento del Backbone y Nuevo Head MLP

Cargamos los pesos pre-entrenados del mejor modelo, congelamos el backbone (feature extractor) para preservar las features aprendidas, y reemplazamos el head lineal simple (`Linear(25, 7)`) por un **MLP más profundo** con BatchNorm, ReLU y Dropout.

## 3. Fase 1: Entrenamiento del Backbone Base

Entrenamos un modelo con una **cabeza densa lineal temporal** (`Linear(25, 7)`) desde cero. El objetivo aquí no es la clasificación perfecta, sino forzar a que las capas convolucionales (el backbone) aprendan buenos extractores de características de las 7 enfermedades.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        # Para accuracy
        _, preds = torch.max(logits, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': loss.item()})

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device, is_test=False):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(logits, dim=1)
            _, preds = torch.max(logits, 1)

            all_probs.append(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)

    # Metricas
    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    # AUC Multi-class (OVR)
    # Nota: Si una clase no está presente en el batch/val_set, ScikitLearn dará error.
    # Usamos multi_class='ovr' (One-vs-Rest)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except ValueError:
        auc = 0.0 # Por si ocurre un fallo en la validación inicial sin todas las clases

    return epoch_loss, acc, f1_macro, auc

In [ ]:
import torch.nn.functional as F
# Focal Loss multiclase (gamma reduce peso de ejemplos fáciles; pesos por clase para minoritarias)
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight = weight  # (num_classes,) opcional
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=None, reduction='none')
        p = F.softmax(logits, dim=1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal = (1 - p_t) ** self.gamma * ce
        if self.weight is not None:
            alpha_t = self.weight.gather(0, targets)
            focal = alpha_t * focal
        return focal.mean() if self.reduction == 'mean' else focal

In [ ]:
print("\n=== FASE 1: Entrenamiento de Extractor Base - Optimización F1 ===")

# 1. Instanciar temporalmente el modelo con cabeza lineal simple
model.classifier = nn.Linear(25, 7).to(device)

# 2. Configuración de Loss: Focal Loss + pesos suavizados + nudge reforzado (akiec, df)
class_weights_smooth = torch.tensor(np.sqrt(class_weights_np), dtype=torch.float32).to(device)
class_weights_smooth = class_weights_smooth / class_weights_smooth.mean()
# Nudge reforzado para las clases que más fallan (akiec=0, df=3)
class_weights_smooth[0] *= 1.8
class_weights_smooth[3] *= 2.5
class_weights_smooth = class_weights_smooth / class_weights_smooth.mean()

criterion = FocalLoss(weight=class_weights_smooth, gamma=2.0)

optimizer_base = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_base = optim.lr_scheduler.CosineAnnealingLR(optimizer_base, T_max=100, eta_min=1e-6)

NUM_EPOCHS_BASE = 100
PATIENCE_BASE = 20
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_lesion_classifier_kan.pth")

# --- VARIABLES DE CONTROL BASADAS EN F1 ---
best_val_f1_base = 0.0
epochs_no_improve_base = 0

for epoch in range(NUM_EPOCHS_BASE):
    print(f"\n[Fase 1] Epoch {epoch+1}/{NUM_EPOCHS_BASE}")

    # Entrenamiento y Evaluación
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_base, device)
    val_loss, val_acc, val_f1, val_auc = evaluate(model, val_loader, criterion, device)

    scheduler_base.step()

    print(f"[Train] Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"[Val]   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: **{val_f1:.4f}** | AUC: {val_auc:.4f}")

    # --- LÓGICA DE GUARDADO Y EARLY STOPPING BASADA EN F1 ---
    if val_f1 > best_val_f1_base:
        best_val_f1_base = val_f1
        epochs_no_improve_base = 0

        # Guardamos el modelo que mejor F1 ha logrado
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"*** Mejor modelo base guardado (F1: {best_val_f1_base:.4f}) ***")
    else:
        epochs_no_improve_base += 1
        print(f"Paciencia: {epochs_no_improve_base}/{PATIENCE_BASE}")

        if epochs_no_improve_base >= PATIENCE_BASE:
            print(f"\n*** Early Stopping Base en época {epoch+1} por falta de mejora en F1 ***")
            break

print(f"\nFase 1 completada. Backbone optimizado para F1 disponible en: {BEST_MODEL_PATH}")


=== FASE 1: Entrenamiento de Extractor Base - Optimización F1 ===

[Fase 1] Epoch 1/100


[Train] Loss: 0.3099 | Acc: 0.6168
[Val]   Loss: 0.2750 | Acc: 0.6610 | F1: **0.3770** | AUC: 0.8480
*** Mejor modelo base guardado (F1: 0.3770) ***

[Fase 1] Epoch 2/100


[Train] Loss: 0.2695 | Acc: 0.6354
[Val]   Loss: 0.2818 | Acc: 0.5165 | F1: **0.3233** | AUC: 0.8271
Paciencia: 1/20

[Fase 1] Epoch 3/100


[Train] Loss: 0.2511 | Acc: 0.6421
[Val]   Loss: 0.2658 | Acc: 0.4606 | F1: **0.3635** | AUC: 0.8494
Paciencia: 2/20

[Fase 1] Epoch 4/100


[Train] Loss: 0.2424 | Acc: 0.6498
[Val]   Loss: 0.2440 | Acc: 0.5553 | F1: **0.3613** | AUC: 0.8553
Paciencia: 3/20

[Fase 1] Epoch 5/100


...

[Train] Loss: 0.1151 | Acc: 0.7532
[Val]   Loss: 0.2183 | Acc: 0.6700 | F1: **0.4332** | AUC: 0.8942
Paciencia: 16/20

[Fase 1] Epoch 48/100


[Train] Loss: 0.1113 | Acc: 0.7487
[Val]   Loss: 0.2159 | Acc: 0.6720 | F1: **0.4639** | AUC: 0.8861
Paciencia: 17/20

[Fase 1] Epoch 49/100


[Train] Loss: 0.1090 | Acc: 0.7455
[Val]   Loss: 0.2242 | Acc: 0.7069 | F1: **0.4992** | AUC: 0.9065
Paciencia: 18/20

[Fase 1] Epoch 50/100


[Train] Loss: 0.1054 | Acc: 0.7612
[Val]   Loss: 0.2651 | Acc: 0.7218 | F1: **0.5234** | AUC: 0.8999
Paciencia: 19/20

[Fase 1] Epoch 51/100


[Train] Loss: 0.0989 | Acc: 0.7665
[Val]   Loss: 0.2311 | Acc: 0.7079 | F1: **0.4768** | AUC: 0.8957
Paciencia: 20/20

*** Early Stopping Base en época 51 por falta de mejora en F1 ***

Fase 1 completada. Backbone optimizado para F1 disponible en: /content/drive/MyDrive/Clasificacion_Lesiones_Dermatologicas/models/best_lesion_classifier_kan.pth


## 4. Fase 2: Educar el KAN Head (Congelando Backbone)

Cargamos los pesos que acabamos de aprender, reemplazamos la cabeza lineal tonta por nuestro **RobustHead con KAN**, y congelamos todo excepto el KAN. Así el KAN aprende a descifrar las 25 features sin destruir el extractor.

In [ ]:
print("\n=== FASE 2: Entrenando Head KAN (Backbone Congelado) - Optimización F1 ===")

# 1. Instanciar la cabeza definitiva KAN
model.classifier = RobustHead(in_features=25, num_classes=7).to(device)

# 2. Cargar SOLO los pesos del backbone
state_dict = torch.load(BEST_MODEL_PATH, map_location=device)
backbone_weights = {k: v for k, v in state_dict.items() if 'classifier' not in k}
model.load_state_dict(backbone_weights, strict=False)
print(f"Backbone de la Fase 1 cargado exitosamente ({len(backbone_weights)} tensores)")

# 3. Congelar el backbone y liberar la cabeza KAN
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables en Fase 2 (RobustHead): {trainable:,}")

# Optimizador y Scheduler
optimizer_kan = optim.Adam(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_kan = optim.lr_scheduler.CosineAnnealingLR(optimizer_kan, T_max=100, eta_min=1e-6)

NUM_EPOCHS_KAN = 100
PATIENCE_KAN = 15
# Cambiamos el nombre del archivo para identificar que es por F1
            # GUARDAMOS EN EL MISMO ARCHIVO

# --- VARIABLES DE CONTROL BASADAS EN F1 ---
best_val_f1_kan = 0.5403
epochs_no_improve_kan = 0

for epoch in range(NUM_EPOCHS_KAN):
    print(f"\n[Fase 2] Epoch {epoch+1}/{NUM_EPOCHS_KAN}")

    # Entrenamiento y Evaluación
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_kan, device)
    val_loss, val_acc, val_f1, val_auc = evaluate(model, val_loader, criterion, device)

    scheduler_kan.step()

    print(f"[Train] Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"[Val]   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1-Score: **{val_f1:.4f}** | AUC: {val_auc:.4f}")

    # --- LÓGICA DE GUARDADO Y EARLY STOPPING BASADA EN F1 ---
    if val_f1 > best_val_f1_kan:
        best_val_f1_kan = val_f1
        epochs_no_improve_kan = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"*** ¡Nuevo récord F1! Modelo guardado (F1: {best_val_f1_kan:.4f}) ***")
    else:
        epochs_no_improve_kan += 1
        print(f"Paciencia: {epochs_no_improve_kan}/{PATIENCE_KAN}")

        if epochs_no_improve_kan >= PATIENCE_KAN:
            print(f"\n*** Early Stopping KAN activado: El F1 no mejora tras {PATIENCE_KAN} épocas ***")
            break


=== FASE 2: Entrenando Head KAN (Backbone Congelado) - Optimización F1 ===
Backbone de la Fase 1 cargado exitosamente (72 tensores)
Parámetros entrenables en Fase 2 (RobustHead): 78,095

[Fase 2] Epoch 1/100


[Train] Loss: 0.4156 | Acc: 0.4363
[Val]   Loss: 0.2765 | Acc: 0.6510 | F1-Score: **0.3747** | AUC: 0.8205
Paciencia: 1/15

[Fase 2] Epoch 2/100


[Train] Loss: 0.2584 | Acc: 0.5866
[Val]   Loss: 0.2275 | Acc: 0.6700 | F1-Score: **0.4564** | AUC: 0.8726
Paciencia: 2/15

[Fase 2] Epoch 3/100


[Train] Loss: 0.2136 | Acc: 0.6259
[Val]   Loss: 0.2137 | Acc: 0.6849 | F1-Score: **0.4804** | AUC: 0.8852
Paciencia: 3/15

[Fase 2] Epoch 4/100


[Train] Loss: 0.1885 | Acc: 0.6556
[Val]   Loss: 0.2130 | Acc: 0.6899 | F1-Score: **0.4851** | AUC: 0.8911
Paciencia: 4/15

[Fase 2] Epoch 5/100


[Train] Loss: 0.1854 | Acc: 0.6571
[Val]   Loss: 0.2093 | Acc: 0.6820 | F1-Score: **0.4764** | AUC: 0.8941
Paciencia: 5/15

[Fase 2] Epoch 6/100


[Train] Loss: 0.1776 | Acc: 0.6616
[Val]   Loss: 0.2036 | Acc: 0.6859 | F1-Score: **0.4916** | AUC: 0.8951
Paciencia: 6/15

[Fase 2] Epoch 7/100


[Train] Loss: 0.1686 | Acc: 0.6702
[Val]   Loss: 0.2078 | Acc: 0.6929 | F1-Score: **0.4922** | AUC: 0.8952
Paciencia: 7/15

[Fase 2] Epoch 8/100


KeyboardInterrupt: 

## 6. Evaluación Final y Gráficas de Aprendizaje

Evaluamos el mejor modelo guardado (`BEST_MODEL_PATH` resultante de las 3 fases) en el conjunto de Test. Además, analizamos el reporte de clasificación, la matriz de confusión y el historial de aprendizaje (Train vs Val Loss, etc) a lo largo del tiempo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print("=== ESTADÍSTICAS FINALES DEL MODELO EN DATASET TEST ===")

# 1. Cargar el MEJOR modelo guardado (compatible con Fase 1 o Fase 2/3)
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_lesion_classifier_kan.pth")
state_dict = torch.load(BEST_MODEL_PATH, map_location=device)
# Si el checkpoint es de Fase 1 (cabeza lineal), el state_dict tiene "classifier.weight";
# si es de Fase 2/3 (RobustHead/KAN), tiene "classifier.fc_pre.0.weight"
is_fase1_checkpoint = "classifier.weight" in state_dict and "classifier.fc_pre.0.weight" not in state_dict
if is_fase1_checkpoint:
    model.classifier = nn.Linear(25, 7).to(device)
    print("(Checkpoint de Fase 1 detectado: se usa cabeza lineal para cargar pesos.)")
model.load_state_dict(state_dict, strict=True)
model.eval()

# 2. Evaluar en TEST
test_loss, test_acc, test_f1, test_auc = evaluate(model, test_loader, criterion, device, is_test=True)
print(f"\nTest Accuracy: {test_acc:.4f} (Objetivo min: 0.69)")
print(f"Test F1-Macro: {test_f1:.4f} (Objetivo min: 0.65)")
print(f"Test AUC:      {test_auc:.4f} (Objetivo min: 0.90)")

# 3. Obtener las predicciones y generar métricas extra
class_names = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'vasc', 'mel']
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test Predictions"):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        _, preds = torch.max(logits, 1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.vstack(all_probs)

print("\n===== Reporte de Clasificación por Clase =====")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

# 4. Graficar Matriz de Confusión
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Clase Predicha')
plt.ylabel('Clase Verdadera')
plt.title('Matriz de Confusión - Mejor Modelo KAN')
plt.show()

# 5. Historial de Pérdidas (Si lo tenemos)
try:
    plt.figure(figsize=(15, 5))

    # Juntamos las métricas de las 3 fases
    combined_train_loss = history_fase1['train_loss'] + history_fase2['train_loss'] + history_fase3['train_loss']
    combined_val_loss = history_fase1['val_loss'] + history_fase2['val_loss'] + history_fase3['val_loss']
    combined_val_f1 = history_fase1['val_f1'] + history_fase2['val_f1'] + history_fase3['val_f1']

    plt.subplot(1, 2, 1)
    plt.plot(combined_train_loss, label='Train Loss')
    plt.plot(combined_val_loss, label='Val Loss')

    # Líneas verticales demarcando fases
    ep1 = len(history_fase1['train_loss'])
    ep2 = ep1 + len(history_fase2['train_loss'])
    plt.axvline(x=ep1, color='r', linestyle='--', label='Start Fase 2 (Head)')
    plt.axvline(x=ep2, color='g', linestyle='--', label='Start Fase 3 (Tuning)')

    plt.title('Entrenamiento vs Validación (Loss)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (CrossEntropy Suavizado)')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(combined_val_f1, label='Val F1-Macro', color='orange')
    plt.axvline(x=ep1, color='r', linestyle='--')
    plt.axvline(x=ep2, color='g', linestyle='--')
    plt.title('Evolución del F1-Score en Validación')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.show()

except NameError:
    print("(El gráfico del historial de épocas estará disponible después de correr las Fases 1-3 en esta sesión).")